# 🌾 AgriMarket Optimizer — Training Demo
**Team 404** | Hackathon Submission

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/huggingface/NSRexe/agrimarket-optimizer-v1/blob/main/notebooks/demo.ipynb)

This notebook trains a Q-Learning agent on all three AgriMarket tasks and visualises the learning curves.

**Tasks:**
- 🍅 **Task 1** — No-Rot: sell perishables before they rot
- 💰 **Task 2** — Profit $1000+: exploit dynamic prices
- ⚠️ **Task 3** — Crash Management: heed market crash warnings

In [ ]:
# ── Step 1: Clone repo from Hugging Face ──────────────────────────────────────
import os

REPO = "NSRexe/agrimarket-optimizer-v1"
LOCAL = "agrimarket-optimizer-v1"

if not os.path.exists(LOCAL):
    !git clone https://huggingface.co/{REPO} {LOCAL}

os.chdir(LOCAL)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Step 2: Install dependencies ──────────────────────────────────────────────
!pip install -q numpy matplotlib pandas

In [ ]:
# ── Step 3: Imports ────────────────────────────────────────────────────────────
import sys
import numpy as np
import matplotlib.pyplot as plt
import os

from env import AgriMarketEnv
from agent import QLearningAgent
from train import run_episode, evaluate

os.makedirs('plots', exist_ok=True)
np.random.seed(42)
print('✅ Setup complete.')

## Task 1 — Basic Sales (Fixed Prices)
The agent learns urgency: sell tomatoes first (rot in 3 days), then corn, then wheat.

**Target:** Zero-rot rate ≥ 90%

In [ ]:
env1 = AgriMarketEnv(task='task1', seed=42)
agent1 = QLearningAgent(epsilon_decay=0.995)

rewards1, profits1, rots1 = [], [], []

for ep in range(1000):
    m = run_episode(env1, agent1, training=True)
    rewards1.append(m['total_reward'])
    profits1.append(m['total_profit'])
    rots1.append(m['rot_events'])

print('Task 1 training complete.')
print(f'Final 100-ep avg reward: {np.mean(rewards1[-100:]):.2f}')
print(f'Final 100-ep avg rots:   {np.mean(rots1[-100:]):.2f}')

In [ ]:
def smooth(x, w=50):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Task 1 — Basic Sales Training Curve', fontsize=13)

axes[0].plot(smooth(rewards1), color='steelblue')
axes[0].set_title('Smoothed Reward')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rots1, alpha=0.3, color='tomato')
axes[1].plot(smooth(rots1), color='darkred', linewidth=2)
axes[1].set_title('Rot Events per Episode')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('# Rots')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/task1_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
eval1 = evaluate('task1', agent1, n_episodes=100)

## Task 2 — Profit Maximization (Dynamic Prices)
Prices change daily — the agent must time its sells to maximize profit.

**Target:** Profit ≥ $1000 in ≥ 70% of episodes

In [ ]:
env2 = AgriMarketEnv(task='task2', seed=42)
agent2 = QLearningAgent(epsilon_decay=0.997)

rewards2, profits2, rots2 = [], [], []

for ep in range(2000):
    m = run_episode(env2, agent2, training=True)
    rewards2.append(m['total_reward'])
    profits2.append(m['total_profit'])
    rots2.append(m['rot_events'])

print('Task 2 training complete.')
print(f'Final 100-ep avg profit: ${np.mean(profits2[-100:]):.2f}')
rate = np.mean([p >= 1000 for p in profits2[-100:]])
print(f'$1000+ success rate (last 100): {rate*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Task 2 — Profit Maximization Training Curve', fontsize=13)

axes[0].plot(smooth(profits2), color='forestgreen')
axes[0].axhline(1000, color='red', linestyle='--', alpha=0.7, label='$1000 target')
axes[0].set_title('Smoothed Profit')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Profit ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(smooth(rewards2), color='steelblue')
axes[1].set_title('Smoothed Reward')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Total Reward')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/task2_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
eval2 = evaluate('task2', agent2, n_episodes=100)

## Task 3 — Risk Management (Crash Events)
Random market crashes can wipe profits — the agent must heed crash warnings and sell before they hit.

**Target:** Crash warning heed rate ≥ 80%

In [ ]:
env3 = AgriMarketEnv(task='task3', seed=42)
agent3 = QLearningAgent(epsilon_decay=0.998)

rewards3, profits3, rots3 = [], [], []
crash_heed_rates = []

for ep in range(3000):
    m = run_episode(env3, agent3, training=True)
    rewards3.append(m['total_reward'])
    profits3.append(m['total_profit'])
    rots3.append(m['rot_events'])
    if m['crash_warnings_received'] > 0:
        crash_heed_rates.append(
            m['crash_warnings_heeded'] / m['crash_warnings_received']
        )

print('Task 3 training complete.')
print(f'Final 100-ep avg profit: ${np.mean(profits3[-100:]):.2f}')
if crash_heed_rates:
    print(f'Crash heed rate (last 100 crash eps): {np.mean(crash_heed_rates[-100:])*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Task 3 — Risk Management Training Curve', fontsize=13)

axes[0].plot(smooth(profits3), color='forestgreen')
axes[0].axhline(1000, color='red', linestyle='--', alpha=0.7, label='$1000 target')
axes[0].set_title('Smoothed Profit')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Profit ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(smooth(rewards3), color='steelblue')
axes[1].set_title('Smoothed Reward')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Total Reward')
axes[1].grid(True, alpha=0.3)

if crash_heed_rates:
    axes[2].plot(smooth(crash_heed_rates, w=min(50, len(crash_heed_rates)//2 or 1)), color='darkorange')
    axes[2].axhline(0.8, color='red', linestyle='--', alpha=0.7, label='80% target')
    axes[2].set_title('Crash Warning Heed Rate')
    axes[2].set_xlabel('Crash Episode')
    axes[2].set_ylabel('Heed Rate')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/task3_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
eval3 = evaluate('task3', agent3, n_episodes=100)

## 📊 Performance Comparison Table

In [ ]:
!pip install pandas -q
import pandas as pd

rows = []
for task, result in [('task1', eval1), ('task2', eval2), ('task3', eval3)]:
    cr = f"{result['crash_heed_rate']*100:.1f}%" if result['crash_heed_rate'] is not None else 'N/A'
    rows.append({
        'Task': task,
        'Mean Profit': f"${result['mean_profit']:.2f}",
        'Zero-Rot Rate': f"{result['zero_rot_rate']*100:.1f}%",
        '$1000+ Rate': f"{result['profit_1000_rate']*100:.1f}%",
        'Crash Heed Rate': cr,
    })

df = pd.DataFrame(rows).set_index('Task')
display(df)

## 🎮 Sample Episode Walkthrough (Task 3)
Watch the agent make decisions step by step.

In [ ]:
from train import demo_episode
demo_episode('task3', agent3, render=True)